# 04 – Kiểm tra chunks và train


In [7]:
from __future__ import annotations

import json
from pathlib import Path
from collections import Counter

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = d:\TTTN-VaxCare-2026\vaxcare-ai


## 1. Kiểm tra chunks.jsonl

In [8]:
chunks_path = PROJECT_ROOT / "knowledge" / "chunks" / "chunks.jsonl"
REQUIRED_FIELDS = {"chunk_id", "doc_id", "title", "text", "source_file", "tier", "vaccine_ids", "page"}

chunks = []
with open(chunks_path, encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            chunks.append(obj)
        except json.JSONDecodeError as e:
            print(f"JSON lỗi dòng {i}: {e}")

print(f"Số chunk: {len(chunks)}")

# Kiểm tra trường
missing_field_count = 0
for c in chunks:
    miss = REQUIRED_FIELDS - set(c.keys())
    if miss:
        missing_field_count += 1
        if missing_field_count <= 3:
            print("Thiếu trường:", miss, "chunk_id=", c.get("chunk_id"))
print("Số chunk thiếu trường:", missing_field_count)

# Trùng chunk_id
ids = [c.get("chunk_id") for c in chunks]
dup = [k for k, v in Counter(ids).items() if v > 1]
print("chunk_id trùng:", len(dup), dup[:5] if dup else "")

# Phân bố doc_id
print("\nTheo doc_id:")
print(Counter(c.get("doc_id") for c in chunks))

# Text rỗng / quá ngắn
short = [c["chunk_id"] for c in chunks if len((c.get("text") or "").strip()) < 30]
print("Chunk text < 30 ký tự:", len(short))

# Ghi lại bản sạch (dedupe theo chunk_id, giữ bản đầu)
seen = set()
clean_chunks = []
for c in chunks:
    cid = c.get("chunk_id")
    if cid in seen:
        continue
    seen.add(cid)
    # đảm bảo vaccine_ids là list
    if not isinstance(c.get("vaccine_ids"), list):
        c["vaccine_ids"] = c.get("vaccine_ids") or []
    clean_chunks.append(c)

out_clean = PROJECT_ROOT / "knowledge" / "chunks" / "chunks.jsonl"
with open(out_clean, "w", encoding="utf-8") as f:
    for c in clean_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print(f"\nĐã ghi lại {len(clean_chunks)} chunk sạch → {out_clean}")

Số chunk: 100
Số chunk thiếu trường: 0
chunk_id trùng: 0 

Theo doc_id:
Counter({'6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em': 51, 'db-protocol': 14, 'tt-10-2024': 13, 'db-vaccine': 13, 'nd13-2024': 9})
Chunk text < 30 ký tự: 0

Đã ghi lại 100 chunk sạch → d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\chunks\chunks.jsonl


## 2. Tạo train.csv / test.csv từ seed_expanded

In [9]:
seed_path = PROJECT_ROOT / "intent_data" / "seed_expanded.csv"
if not seed_path.exists():
    raise FileNotFoundError("Chạy notebook 28 trước để tạo seed_expanded.csv")

df = pd.read_csv(seed_path)
print("Input:", len(df))
print(df["label"].value_counts())

# Shuffle + split 80/20 theo từng label (stratified đơn giản)
train_rows = []
test_rows = []
for label, g in df.groupby("label"):
    g = g.sample(frac=1.0, random_state=42)
    n_test = max(5, int(len(g) * 0.15))
    test_rows.append(g.iloc[:n_test])
    train_rows.append(g.iloc[n_test:])

train_df = pd.concat(train_rows).sample(frac=1.0, random_state=42).reset_index(drop=True)
test_df = pd.concat(test_rows).sample(frac=1.0, random_state=42).reset_index(drop=True)

# Chỉ giữ text, label cho file train chính thức
train_out = train_df[["text", "label"]].copy()
test_out = test_df[["text", "label"]].copy()

train_path = PROJECT_ROOT / "intent_data" / "train.csv"
test_path = PROJECT_ROOT / "intent_data" / "test.csv"
train_out.to_csv(train_path, index=False, encoding="utf-8-sig")
test_out.to_csv(test_path, index=False, encoding="utf-8-sig")

print("\ntrain.csv:", len(train_out))
print(train_out["label"].value_counts())
print("\ntest.csv:", len(test_out))
print(test_out["label"].value_counts())

Input: 482
label
faq_vaccine        195
vaxcare_lookup     111
schedule_advice     83
history_lookup      63
chitchat            15
out_of_scope        15
Name: count, dtype: int64

train.csv: 406
label
faq_vaccine        166
vaxcare_lookup      95
schedule_advice     71
history_lookup      54
out_of_scope        10
chitchat            10
Name: count, dtype: int64

test.csv: 76
label
faq_vaccine        29
vaxcare_lookup     16
schedule_advice    12
history_lookup      9
chitchat            5
out_of_scope        5
Name: count, dtype: int64


In [10]:
# README quy ước gán nhãn (nếu chưa có)
readme = PROJECT_ROOT / "intent_data" / "README.md"
if not readme.exists():
    readme.write_text(
        """# Intent data – quy ước gán nhãn

6 nhóm intent (cột `label`):

| label | Mô tả | Ví dụ |
|-------|--------|--------|
| faq_vaccine | Hỏi công dụng, chống chỉ định, phác đồ, lưu ý vaccine | "Vaccine HPV phòng bệnh gì?" |
| schedule_advice | Hỏi mốc / lộ trình theo tuổi hoặc năm sinh | "Tôi 20 tuổi nên kiểm tra vaccine nào?" |
| history_lookup | Hỏi / đối chiếu hồ sơ tiêm đã ghi nhận | "Tôi đã tiêm mũi 1 rồi, tiếp theo?" |
| vaxcare_lookup | Tìm vaccine, cơ sở, đặt lịch trên VaxCare | "Đặt lịch tiêm MMR" |
| chitchat | Chào hỏi, cảm ơn, ngoài nội dung chuyên môn nhẹ | "Xin chào", "Cảm ơn" |
| out_of_scope | Cấp cứu, chẩn đoán, kê đơn, ngoài phạm vi | "Đau ngực khó thở phải làm sao?" |

Quy tắc:
- Mỗi câu chỉ 1 nhãn chính.
- Nếu vừa hỏi kiến thức vừa đặt lịch → ưu tiên intent rõ ràng hơn trong câu (thường faq hoặc vaxcare).
- Câu cấp cứu / triệu chứng nặng → luôn `out_of_scope`.

File:
- `seed_expanded.csv`: seed + nhân rộng (có cột source)
- `train.csv` / `test.csv`: text,label dùng train sklearn
""",
        encoding="utf-8",
    )
    print("Created", readme)
else:
    print("README already exists")

README already exists


## 3. Khung pytest cho schedule_engine

File: `tests/test_schedule_engine.py`  
Hiện tại test loader + logic “expected vs records” đơn giản. Khi có `schedule_engine.py` đầy đủ sẽ mở rộng.

In [11]:
test_file = PROJECT_ROOT / "tests" / "test_schedule_engine.py"
print("Test file exists:", test_file.exists())
print(test_file)
# Nội dung đã được tạo sẵn trong repo (xem cell markdown / file trên disk).
# Chạy: pytest tests/test_schedule_engine.py -q

Test file exists: True
d:\TTTN-VaxCare-2026\vaxcare-ai\tests\test_schedule_engine.py


In [12]:
# Chạy thử pytest nếu môi trường có
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "pytest", str(PROJECT_ROOT / "tests" / "test_schedule_engine.py"), "-q"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
)
print(r.stdout)
print(r.stderr)
print("returncode", r.returncode)

.......                                                                  [100%]
7 passed in 2.02s


returncode 0
